# Month 3 — Cross-Dataset External Validation: Combined 4-Site Cohort vs. Framingham

This notebook implements Month 3 of the research framework:
1. **Data Harmonization**: Aligning the **Combined 4-Site UCI Dataset** (Cleveland, Hungarian, Switzerland, VA Long Beach; N=920) and the **Framingham Heart Study dataset** (N=4,240) onto a unified 5-feature schema (`age`, `sex`, `sysBP`, `totChol`, `diabetes`, target `target`).
2. **External Validation Pipeline**: Training the full tuned ensemble framework on the complete harmonized Combined 4-site dataset, then evaluating performance directly on the unseen complete Framingham dataset (4,240 instances).
3. **3-Way Performance Comparison**: Comparing **Cleveland-Only Nested CV** (old baseline) vs. **Combined 4-Site Nested CV** (new baseline) vs. **Framingham External Validation** to evaluate whether multi-site training data improves external generalization compared to single-site training data.


In [1]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Ensure src module is in python path
sys.path.append('..')

from src.preprocessing import load_combined_dataset
from src.data_harmonization import load_framingham_raw, harmonize_datasets
from src.external_validation import run_cross_dataset_validation

# Load configuration
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully.")


Configuration loaded successfully.


## 1. Load and Harmonize Combined 4-Site UCI & Framingham Datasets
Load raw datasets, align schema onto common attributes (`age`, `sex`, `sysBP`, `totChol`, `diabetes`), and binarize target variables.


In [2]:
results_dir = '../' + config['results_dir']
df_comb_raw = load_combined_dataset('../data', results_dir)
df_fram_raw = load_framingham_raw('../data/raw_framingham.csv')

print(f"Combined 4-Site Dataset Shape: {df_comb_raw.shape}")
print(f"Framingham Dataset Shape: {df_fram_raw.shape}")

c_harm, f_harm = harmonize_datasets(df_comb_raw, df_fram_raw)

print("\nHarmonized Combined UCI Dataset Shape:", c_harm.shape)
print("Harmonized Framingham Dataset Shape:", f_harm.shape)

print("\nHarmonized Schema Columns:", c_harm.columns.tolist())


Combined 4-Site Dataset Shape: (920, 15)
Framingham Dataset Shape: (4240, 16)

Harmonized Combined UCI Dataset Shape: (920, 6)
Harmonized Framingham Dataset Shape: (4240, 6)

Harmonized Schema Columns: ['age', 'sex', 'sysBP', 'totChol', 'diabetes', 'target']


### Documentation of Feature Loss and Simplification
- **Features dropped from Combined UCI** (missing in Framingham): `cp` (chest pain), `restecg`, `thalach` (max heart rate), `exang`, `oldpeak`, `slope`, `ca` (vessels), `thal`.
- **Features dropped from Framingham** (missing in UCI): `education`, `currentSmoker`, `cigsPerDay`, `BPMeds`, `prevalentStroke`, `prevalentHyp`, `diaBP`, `BMI`, `heartRate`, `glucose`.
- **Target Definitions**:
  - *Combined UCI*: Angiographic disease status (`target > 0`).
  - *Framingham*: 10-year risk of coronary heart disease (`TenYearCHD`).


## 2. Execute 3-Way Cross-Dataset External Validation
Train tuned classifiers (RF, XGBoost, AdaBoost, Soft Ensemble) on full Combined 4-site harmonized dataset (N=920) and evaluate on the unseen Framingham dataset (N=4,240). Also compute Cleveland-only CV and Combined 4-site CV on the 5-feature harmonized schema.


In [3]:
print("Executing 3-way cross-dataset validation pipeline...")
validation_results = run_cross_dataset_validation(
    c_harm,
    f_harm,
    target_col='target',
    outer_splits=config['cv']['outer_folds'],
    inner_splits=config['cv']['inner_folds'],
    n_trials=config['cv']['optuna_n_trials'],
    random_state=config['random_state']
)

comp_df = validation_results['comparison_table']
print("\n=== 3-Way Performance Transferability Comparison ===")
display(comp_df)


Executing 3-way cross-dataset validation pipeline...
Running Cleveland-only nested CV on harmonized schema...


Running Combined 4-site nested CV on harmonized schema...


Fitting preprocessor and tuning hyper-parameters on full combined 4-site dataset...


Evaluating trained pipeline on unseen Framingham dataset...



=== 3-Way Performance Transferability Comparison ===


,Model,Cleveland CV Acc,Combined 4-Site CV Acc,Framingham Ext Acc,Cleveland CV ROC-AUC,Combined 4-Site CV ROC-AUC,Framingham Ext ROC-AUC,Framingham Ext F1,Framingham Ext PR-AUC
0,RANDOM_FOREST,0.6702,0.6717,0.7156,0.7190,0.7445,0.6867,0.3424,0.2703
1,XGBOOST,0.6733,0.6902,0.7264,0.7162,0.7473,0.6938,0.3483,0.2655
2,ADABOOST,0.6533,0.6967,0.7439,0.7055,0.7465,0.6839,0.3313,0.2725
3,ENSEMBLE,0.6766,0.6783,0.7278,0.7273,0.7471,0.6920,0.3473,0.2722


## 3. Confusion Matrices on External Framingham Dataset
Plot confusion matrices across all models evaluated on the external Framingham cohort.


In [4]:
fram_metrics = validation_results['framingham_external']

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

model_names = ['random_forest', 'xgboost', 'adaboost', 'ensemble']

for idx, m_name in enumerate(model_names):
    m_data = fram_metrics[m_name]
    cm = np.array(m_data['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[idx],
                xticklabels=['No CHD', '10-Yr CHD'],
                yticklabels=['No CHD', '10-Yr CHD'])
    axes[idx].set_title(f"Framingham Confusion Matrix: {m_name.upper()}")
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(f"{results_dir}/framingham_external_confusion_matrices.png", dpi=300)
plt.close('all')
print("Saved updated Framingham external confusion matrices plot.")


Saved updated Framingham external confusion matrices plot.
